In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to cache file
cache_file = "cache_all_slides.pkl"

# Path to zarr
local_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
from helper_functions import subset_df, subset_df_list

df_HE = subset_df(df_all, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
df_HE = subset_df_list(df_HE, "T_category", "Placenta, Fetal Membranes, and Fetus")

In [ ]:
all_filenames = df_HE["filename"].tolist()
unique_wsi = set(all_filenames)

print("Number of unique wsi filenames: ",  len(unique_wsi))
print("Number of wsi filenames: ", len(all_filenames))

In [ ]:
feature_result = r"D:\NOTEBOOKS\Christine\all_slides\combined_feature_summary.csv"
df_feature_result = pd.read_csv(feature_result)
df_feature = df_feature_result[df_feature_result['status'] == "feature extraction complete"]["wsi_path"].astype(str)
with_features = set(df_feature)
print("WSIs with features detected: ", len(with_features))
all_filenames = list(with_features)

In [ ]:
# Visualize Single Slide
import os
from wsidata import open_wsi

path = all_filenames[0]
print(path)

zarr_path = os.path.join(local_dir, os.path.basename(path).replace(".mrxs", ".zarr"))
wsi = open_wsi(path, zarr_path)
wsi

In [ ]:
import lazyslide as zs
zs.tl.spatial_domain(
    wsi,
    feature_key="features_conch",  
    tile_key="tiles_224",                
    resolution=0.2,                    
    key_added="domain"                  
)
zs.pl.tiles(wsi, tissue_key= "tissue_default", tile_key="tiles_224", color="domain", alpha=0.5)

In [ ]:
# Concatenate tiles belonging to the same domain into shapes
zs.tl.tile_shaper(
    wsi,
    groupby='domain',       
    tile_key='tiles_224',    
    key_added='domain_shapes'
)